### Build Results Fact

 Read silver `results` table
 Read silver `sprints` table
 Add new column `session_type` with values `RACE` or `SPRINT`
 UNION `results` and `sprints`
 Derive additional columns
  - is_win -> Indicates that the driver own the race
  - is_podium -> Indicates that the driver scored a podium result (1, 2, 3)
  - has_points -> Indicates that the driver has scored points
 Write the transformed data to gold `fact_session_results` table

- silver.results
- sliver.sprints

In [0]:
from pyspark.sql import functions as f

In [0]:
%run ../00-common-Config/01-environment-variable

In [0]:
target_table = f"{catalog_name}.{gold_schema}.fact_session_results"

In [0]:
results_df = spark.table(f"{catalog_name}.{silver_schema}.results")\
    .withColumn("session_type", f.lit("results"))\
        .drop("ingestion_timestamp","race_date","source_file","race_name")


In [0]:
sprints_df = spark.table(f"{catalog_name}.{silver_schema}.sprints")\
    .withColumn("session_type", f.lit("sprints"))\
        .drop("ingestion_timestamp","race_date","source_file","race_name")

In [0]:
result_sprints_df=results_df.unionByName(sprints_df)


In [0]:
result_sprints_df = (
    result_sprints_df
    .withColumn("is_win", f.col("finish_position") == 1)
    .withColumn("is_podium", f.col("finish_position").between(1, 3))
    .withColumn("has_points", f.col("points") > 0)
)
display(result_sprints_df)

In [0]:
(
    result_sprints_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(target_table)
)